# 02 — EDA & Experiment Profiling
**Purpose:** Deep EDA structured around the experiment batching plan. For each dataset addition, measure if it provides new information.

**Key outputs:**
1. Target correlation matrix (Alkalinity, EC, DRP) → informs multi-target decision
2. Per-dataset information gain → validates experiment plan
3. Train vs validation distribution comparison → detect covariate shift

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

INPUT_DIR = '/kaggle/input/ey-water-quality-data'
WORK_DIR = '/kaggle/working'

In [ ]:
# Load enriched datasets from notebook 01
train = pd.read_parquet(f'{WORK_DIR}/train_enriched.parquet')
val = pd.read_parquet(f'{WORK_DIR}/val_enriched.parquet')
print(f'Train: {train.shape}, Val: {val.shape}')

## 1. Target Analysis
### 1a. Target Distributions

In [ ]:
# Identify target columns (adjust based on actual names)
TARGET_COLS = []  # e.g., ['Total_Alkalinity_mg/L', 'Electrical_Conductance_uS/cm', 'Dissolved_Reactive_Phosphorus_ug/L']
# Auto-detect
for col in train.columns:
    cl = col.lower()
    if any(k in cl for k in ['alkalinity', 'conductance', 'phosphorus']):
        TARGET_COLS.append(col)
print(f'Targets: {TARGET_COLS}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(TARGET_COLS):
    ax = axes[i]
    train[col].hist(bins=50, ax=ax, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(f'{col}\nmean={train[col].mean():.1f}, skew={train[col].skew():.2f}')
    ax.axvline(train[col].median(), color='red', linestyle='--', label='median')
    ax.legend()
plt.suptitle('Target Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1b. Target Correlation Matrix (CRITICAL for Multi-Target Decision)

In [ ]:
# This directly informs Exp 4: separate vs multi-target models
target_corr = train[TARGET_COLS].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(target_corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, ax=ax, fmt='.3f', linewidths=2)
ax.set_title('Target Correlation Matrix\n(|ρ| > 0.5 → multi-target worth exploring)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Multi-Target Decision Criteria ===')
for i, c1 in enumerate(TARGET_COLS):
    for c2 in TARGET_COLS[i+1:]:
        r = target_corr.loc[c1, c2]
        verdict = '→ RegressorChain may help' if abs(r) > 0.5 else '→ Separate models likely better'
        print(f'  {c1} ↔ {c2}: ρ={r:.3f} {verdict}')

### 1c. Spatial Patterns

In [ ]:
# Mean water quality per station
LAT_COL = 'Latitude'  # adjust
LON_COL = 'Longitude'  # adjust
STATION_COL = 'GEMS_Station_Number'  # adjust

station_means = train.groupby([STATION_COL, LAT_COL, LON_COL])[TARGET_COLS].mean().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, col in enumerate(TARGET_COLS):
    ax = axes[i]
    sc = ax.scatter(station_means[LON_COL], station_means[LAT_COL], 
                    c=station_means[col], cmap='RdYlGn_r', s=30, alpha=0.8)
    plt.colorbar(sc, ax=ax)
    ax.set_title(col.split('_')[0][:15], fontsize=11)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

# Add validation stations
for ax in axes:
    ax.scatter(val[LON_COL], val[LAT_COL], c='black', marker='x', s=50, label='Validation')
    ax.legend()

plt.suptitle('Spatial Distribution of Water Quality (train=dots, val=X)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Per-Dataset Information Gain
Quick mutual information score between each feature group and each target.

In [ ]:
# Define feature groups by dataset source
# Adjust these based on actual column names
FEATURE_GROUPS = {
    'landsat': [c for c in train.columns if any(k in c.lower() for k in ['b1','b2','b3','b4','b5','b6','b7','ndvi','ndwi','ndbi'])],
    'terraclimate': [c for c in train.columns if any(k in c.lower() for k in ['tmax','tmin','ppt','soil','aet','pet','def','vpd','pdsi','srad','ws','vap','swe'])],
    'elevation': [c for c in train.columns if 'elevation' in c.lower()],
    'soilgrids': [c for c in train.columns if 'soil_' in c.lower()],
    'weather': [c for c in train.columns if any(k in c.lower() for k in ['precip','temp_','wind_'])],
    'osm': [c for c in train.columns if any(k in c.lower() for k in ['mine','wastewater','farmland','road','osm'])],
    'hydroatlas': [c for c in train.columns if 'basin_' in c.lower()],
    'riveratlas': [c for c in train.columns if 'river_' in c.lower()],
}

for group, cols in FEATURE_GROUPS.items():
    print(f'{group}: {len(cols)} features')

In [ ]:
# Compute mutual information per feature group per target
mi_results = {}

for target in TARGET_COLS:
    target_clean = target.split('_')[0][:10]
    mi_results[target_clean] = {}
    
    y = train[target].dropna()
    valid_idx = y.index
    
    for group, cols in FEATURE_GROUPS.items():
        valid_cols = [c for c in cols if c in train.columns]
        if not valid_cols:
            mi_results[target_clean][group] = 0
            continue
        
        X_group = train.loc[valid_idx, valid_cols].fillna(0)
        mi = mutual_info_regression(X_group, y, random_state=42, n_neighbors=5)
        mi_results[target_clean][group] = float(np.mean(mi))

mi_df = pd.DataFrame(mi_results)
print('\n=== Mutual Information by Dataset Group ===')
display(mi_df.sort_values(mi_df.columns[0], ascending=False))

In [ ]:
# Visualize MI heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(mi_df, annot=True, cmap='YlOrRd', fmt='.4f', ax=ax)
ax.set_title('Mutual Information: Feature Group vs Target\n(Higher = More Informative)', 
             fontsize=13, fontweight='bold')
ax.set_ylabel('Dataset Group')
plt.tight_layout()
plt.show()

print('\n=== Experiment Viability ===')
for group in mi_df.index:
    avg_mi = mi_df.loc[group].mean()
    verdict = '✅ Keep' if avg_mi > 0.01 else '⚠️ Low info — may drop'
    print(f'  {group}: avg MI={avg_mi:.4f} {verdict}')

## 3. Train vs Validation Distribution (Covariate Shift)

In [ ]:
from scipy import stats

# KS-test on numeric features
numeric_cols = train.select_dtypes(include=[np.number]).columns
common_cols = [c for c in numeric_cols if c in val.columns and c not in TARGET_COLS]

shift_results = []
for col in common_cols:
    t_vals = train[col].dropna()
    v_vals = val[col].dropna()
    if len(t_vals) > 0 and len(v_vals) > 0:
        ks_stat, p_val = stats.ks_2samp(t_vals, v_vals)
        shift_results.append({'feature': col, 'ks_stat': ks_stat, 'p_value': p_val})

shift_df = pd.DataFrame(shift_results).sort_values('ks_stat', ascending=False)
print('\n=== Top 15 Features with Covariate Shift (high KS = different distribution) ===')
display(shift_df.head(15))

n_shifted = (shift_df['p_value'] < 0.05).sum()
print(f'\n{n_shifted}/{len(shift_df)} features show significant distribution shift (p<0.05)')

## 4. Missing Values Analysis

In [ ]:
null_pcts = train.isnull().mean().sort_values(ascending=False)
null_pcts_nonzero = null_pcts[null_pcts > 0]

if len(null_pcts_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(12, max(4, len(null_pcts_nonzero) * 0.3)))
    null_pcts_nonzero.plot(kind='barh', ax=ax, color='coral')
    ax.set_title('Missing Values by Feature (% null)', fontweight='bold')
    ax.set_xlabel('Fraction Missing')
    plt.tight_layout()
    plt.show()
else:
    print('✅ No missing values in training data')

print(f'\nTotal features with nulls: {len(null_pcts_nonzero)}')
print(f'Features with >50% nulls: {(null_pcts > 0.5).sum()}')

## 5. Summary & Recommendations

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'\n1. TARGET CORRELATIONS:')
print(f'   → Multi-target decision: check correlation matrix above')
print(f'\n2. DATASET INFORMATION GAIN:')
print(f'   → Check MI heatmap — datasets with MI > 0.01 are keepers')
print(f'\n3. COVARIATE SHIFT:')
print(f'   → {n_shifted} features differ between train/val')
print(f'   → This is expected (different regions) — spatial CV handles this')
print(f'\n4. MISSING VALUES:')
print(f'   → {len(null_pcts_nonzero)} features have nulls')
print(f'\n5. NEXT STEPS:')
print(f'   → Proceed to notebook 03 (Feature Engineering)')
print(f'   → Use findings above to guide feature selection')